# wrap-forward-fn-generic — worked example 1: wrap_forward_fn shell: unbox, call, box

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wrap-forward-fn-generic`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`wrap_forward_fn(fwd_fn)` is a closure factory: it returns a new `tensor_func` that unboxes `Tensor` args to their raw `.array`, calls `fwd_fn` on the raw values, and boxes the result back into a `Tensor`. One factory handles every op because only the captured `fwd_fn` varies.

## Worked solution

We build the minimal Tensor-aware wrapper.

1. `wrap_forward_fn` closes over `fwd_fn` and returns `tensor_func(*args, **kwargs)`.
2. Unbox: a comprehension replaces each `Tensor` arg with `.array`; plain ints/floats/raw torch tensors pass through.
3. Call: `fwd_fn(*raw_args, **kwargs)` runs on the raw tensors.
4. Box: we wrap the output in a fresh `Tensor` and return it.

We wrap `t.log` once and apply it to a boxed tensor, printing that the result is a `Tensor` carrying the log values.

In [ ]:
import torch as t

class Tensor:
    def __init__(self, array):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)
    def __repr__(self):
        return f'Tensor({self.array.tolist()})'

def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        return Tensor(out_raw)
    return tensor_func

log = wrap_forward_fn(t.log)
import math
x = Tensor([1.0, math.e, math.e ** 2])
out = log(x)
print('type:', type(out).__name__)
print('values:', [round(v, 4) for v in out.array.tolist()])